For this assignment, we are going to use the Steel Plates Faults Data Set as available from here (https://archive.ics.uci.edu/ml/machine-learning-databases/00198/Faults.NNA (Links to an external site.)). Following are the list of attributes of the dataset.

Type of dependent variables (7 Types of Steel Plates Faults):
- 1.Pastry
- 2.Z_Scratch
- 3.K_Scatch
- 4.Stains
- 5.Dirtiness
- 6.Bumps
- 7.Other_Faults

27 independent variables:
- X_Minimum
- X_Maximum
- Y_Minimum
- Y_Maximum
- Pixels_Areas
- X_Perimeter
- Y_Perimeter
- Sum_of_Luminosity
- Minimum_of_Luminosity
- Maximum_of_Luminosity
- Length_of_Conveyer
- TypeOfSteel_A300
- TypeOfSteel_A400
- Steel_Plate_Thickness
- Edges_Index
- Empty_Index
- Square_Index
- Outside_X_Index
- Edges_X_Index
- Edges_Y_Index
- Outside_Global_Index
- LogOfAreas
- Log_X_Index
- Log_Y_Index
- Orientation_Index
- Luminosity_Index
- SigmoidOfAreas

Among the independent variables only the Steel types (12th and 13th) are categorical variables, rest are numeric. For this exercise use neural network and see how well you could predict the type of faults in steel plates from numeric attributes only. [Note: To save time and energy use the hidden layer numbers, and number of nodes in hidden layers that your computer can handle].

### import packages

In [1]:
import os, glob

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv2D, Flatten, Dense

import warnings
warnings.filterwarnings('ignore')

2023-11-30 16:48:55.002624: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


### read data

In [34]:
cols = ["X_Minimum","X_Maximum","Y_Minimum","Y_Maximum","Pixels_Areas","X_Perimeter","Y_Perimeter",
        "Sum_of_Luminosity","Minimum_of_Luminosity","Maximum_of_Luminosity","Length_of_Conveyer",
        "TypeOfSteel_A300","TypeOfSteel_A400","Steel_Plate_Thickness","Edges_Index",
        "Empty_Index","Square_Index","Outside_X_Index","Edges_X_Index","Edges_Y_Index",
        "Outside_Global_Index","LogOfAreas","Log_X_Index","Log_Y_Index","Orientation_Index",
        "Luminosity_Index","SigmoidOfAreas",
        "Pastry","Z_Scratch","K_Scatch","Stains","Dirtiness","Bumps","Other_Faults"]
df = pd.read_table("https://archive.ics.uci.edu/ml/machine-learning-databases/00198/Faults.NNA", names = cols, header = None)
df.head()

,X_Minimum,X_Maximum,Y_Minimum,Y_Maximum,Pixels_Areas,X_Perimeter,Y_Perimeter,Sum_of_Luminosity,Minimum_of_Luminosity,Maximum_of_Luminosity,...,Orientation_Index,Luminosity_Index,SigmoidOfAreas,Pastry,Z_Scratch,K_Scatch,Stains,Dirtiness,Bumps,Other_Faults
0,42,50,270900,270944,267,17,44,24220,76,108,...,0.8182,-0.2913,0.5822,1,0,0,0,0,0,0
1,645,651,2538079,2538108,108,10,30,11397,84,123,...,0.7931,-0.1756,0.2984,1,0,0,0,0,0,0
2,829,835,1553913,1553931,71,8,19,7972,99,125,...,0.6667,-0.1228,0.2150,1,0,0,0,0,0,0
3,853,860,369370,369415,176,13,45,18996,99,126,...,0.8444,-0.1568,0.5212,1,0,0,0,0,0,0
4,1289,1306,498078,498335,2409,60,260,246930,37,126,...,0.9338,-0.1992,1.0000,1,0,0,0,0,0,0


In [35]:
df.shape

(1941, 34)

### prepare data

In [36]:
#Creating X and y
x_col = [x for x in range(11)]
x_col.extend([x for x in range(13,27)])
X = df.iloc[:, x_col]
y = df.iloc[:, 27:34]
#y = pd.get_dummies(df.iloc[:, 27:34]).idxmax(1)

# Create train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3)

# Scale the data -- one reason is to save on computing later
ss = StandardScaler()
X_train = ss.fit_transform(X_train)
X_test = ss.transform(X_test)

#y_train = tf.keras.utils.to_categorical(y_train, 7)
#y_test = tf.keras.utils.to_categorical(y_test, 7)

In [37]:
print(y_test.shape)

(583, 7)


### build a neural network model

In [38]:
nn = Sequential()
nn.add(Dense(128, activation = 'relu'))
nn.add(Dense(64, activation = 'softmax'))
nn.add(Dense(16, activation='relu'))
nn.add(Dense(7, activation = 'softmax'))


# 'adam' for stochastic gradient descent
nn.compile(optimizer = 'adam', loss = 'categorical_crossentropy', metrics = ['accuracy'])

In [ ]:
#print(X_train.shape)

In [39]:
nn.fit(X_train, y_train)
nn.summary()

43/43 [==============================] - 1s 2ms/step - loss: 1.8892 - accuracy: 0.3189
Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_4 (Dense)             (None, 128)               3328      
                                                                 
 dense_5 (Dense)             (None, 64)                8256      
                                                                 
 dense_6 (Dense)             (None, 16)                1040      
                                                                 
 dense_7 (Dense)             (None, 7)                 119       
                                                                 
Total params: 12743 (49.78 KB)
Trainable params: 12743 (49.78 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [40]:
y_pred = nn.predict(X_test)
y_pred = pd.get_dummies(pd.DataFrame(y_pred, columns = df.columns[27:34])).idxmax(1)  
# idxmax() find the index of the maximum value in an array: if 1 then row ; if 0 then column . 
y_test = pd.get_dummies(y_test).idxmax(1)

19/19 [==============================] - 0s 1ms/step


In [45]:
y_pred

0      Other_Faults
1      Other_Faults
2      Other_Faults
3          K_Scatch
4          K_Scatch
           ...     
578        K_Scatch
579    Other_Faults
580    Other_Faults
581    Other_Faults
582        K_Scatch
Length: 583, dtype: object

In [44]:
y_pred1 = nn.predict(X_test)
y_pred1 

19/19 [==============================] - 0s 2ms/step


array([[0.13927965, 0.13525255, 0.1641546 , ..., 0.12495922, 0.1508104 ,
        0.166749  ],
       [0.14283329, 0.13791388, 0.16755895, ..., 0.12190727, 0.1490128 ,
        0.16908135],
       [0.1404633 , 0.1348042 , 0.1629707 , ..., 0.12361772, 0.15051055,
        0.16843046],
       ...,
       [0.13500139, 0.13175799, 0.16720468, ..., 0.12359838, 0.1548056 ,
        0.1673513 ],
       [0.13312094, 0.1315167 , 0.17198838, ..., 0.12369952, 0.15197612,
        0.17246744],
       [0.12824942, 0.12726991, 0.19945958, ..., 0.12086515, 0.14989978,
        0.16872537]], dtype=float32)

In [41]:
print(y_test.shape)

(583,)


In [42]:
#print(y_test)

In [43]:
print(accuracy_score(y_test,y_pred))
print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))

0.3670668953687822
[[  0   0  71  56   0   0   0]
 [  0   0   4  17   0   0   0]
 [  0   0 117   0   0   0   0]
 [  0   0 104  97   0   0   0]
 [  0   0   9  39   0   0   0]
 [  0   0  16   1   0   0   0]
 [  0   0  33  19   0   0   0]]
              precision    recall  f1-score   support

       Bumps       0.00      0.00      0.00       127
   Dirtiness       0.00      0.00      0.00        21
    K_Scatch       0.33      1.00      0.50       117
Other_Faults       0.42      0.48      0.45       201
      Pastry       0.00      0.00      0.00        48
      Stains       0.00      0.00      0.00        17
   Z_Scratch       0.00      0.00      0.00        52

    accuracy                           0.37       583
   macro avg       0.11      0.21      0.14       583
weighted avg       0.21      0.37      0.26       583

